In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


In [8]:
df = playerScoring('Tyrese Maxey', s26, current_date, teamStarPlayer, projectedStartingFive)
len(df)

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:363: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])


136

### Load Player Data and Bookmaker Data

In [10]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_13589/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,John Collins,Over,14.0,-137,2025-11-21,2025-11-20T21:47:51Z
1,PrizePicks,player_points,John Collins,Under,14.0,-137,2025-11-21,2025-11-20T21:47:51Z
2,PrizePicks,player_points,James Harden,Over,27.5,-137,2025-11-21,2025-11-20T21:47:51Z
3,PrizePicks,player_points,James Harden,Under,27.5,-137,2025-11-21,2025-11-20T21:47:51Z
4,PrizePicks,player_points,Franz Wagner,Over,23.5,-137,2025-11-21,2025-11-20T21:47:51Z


### Update projected starting lineups

In [7]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 8 teams with confirmed lineups


### Top EVs for single bets

In [11]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 58 unique players...
Error getting prediction for Paul George: float division by zero


/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/pipeline.py:364: SettingWithCopyWarning: 
A value is trying to be set on a cop

,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
0,Bobby Portis,FanDuel,14.5,9.42,Under,-104,1,5.41,0.562,High
1,Bobby Portis,BetRivers,13.5,9.42,Under,105,0,4.82,0.459,High
2,Bobby Portis,DraftKings,14.5,9.42,Under,-112,1,4.73,0.530,High
3,Bobby Portis,BetMGM,13.5,9.42,Under,100,0,4.47,0.447,High
4,Bobby Portis,BetRivers,14.5,9.42,Under,-120,1,4.36,0.523,High


## Top EVs for 2 leg bets

### Underdog picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 50 players...
Error getting prediction for Paul George: float division by zero
Processing 44 players with valid predictions...
Generated 861 valid 2-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 41 combinations from 861 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Bobby Portis,Will Richard,14.5,6.5,9.42,11.10,under,over,1,6.74,0.337,High,High
1,Deni Avdija,Will Richard,23.5,6.5,29.23,11.10,over,over,1,6.35,0.318,High,High
2,Bobby Portis,Deni Avdija,14.5,23.5,9.42,29.23,under,over,1,6.01,0.301,High,High
3,Tyus Jones,Dominick Barlow,3.5,5.5,4.46,8.15,over,over,0,4.24,0.212,Low,Med
4,Kobe Sanders,Dominick Barlow,9.5,5.5,12.70,8.15,over,over,0,4.08,0.204,High,Med
5,Tyus Jones,Donovan Clingan,3.5,8.5,4.46,11.46,over,over,0,3.63,0.182,Low,High
6,Kobe Sanders,Donovan Clingan,9.5,8.5,12.70,11.46,over,over,0,3.48,0.174,High,High
7,Cam Spencer,Buddy Hield,12.5,7.5,9.37,9.95,under,over,0,2.72,0.136,High,High
8,Cam Spencer,Brandin Podziemski,12.5,11.5,9.37,14.85,under,over,0,2.66,0.133,High,High
9,Zach Edey,Brandin Podziemski,12.5,11.5,14.88,14.85,over,over,0,2.45,0.122,Med,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 68 players...
Error getting prediction for Paul George: float division by zero
Processing 61 players with valid predictions...
Generated 1669 valid 2-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 60 combinations from 1669 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Bobby Portis,Will Richard,14.5,6.5,9.42,11.10,under,over,1,6.75,0.338,High,High
1,Deni Avdija,Will Richard,23.5,6.5,29.23,11.10,over,over,1,6.26,0.313,High,High
2,Bobby Portis,Deni Avdija,14.5,23.5,9.42,29.23,under,over,1,6.11,0.305,High,High
3,Tyus Jones,Donovan Clingan,3.5,8.5,4.46,11.46,over,over,0,3.57,0.179,Low,High
4,Kobe Sanders,Donovan Clingan,9.5,8.5,12.70,11.46,over,over,0,3.28,0.164,High,High


## 3 leg parlay

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 50 players...
Error getting prediction for Paul George: float division by zero
Processing 44 players with valid predictions...
Generated 12786 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 29 combinations from 12786 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Bobby Portis,Deni Avdija,Will Richard,14.5,23.5,6.5,9.42,29.23,11.10,under,over,over,1,13.02,0.260,High,High,High
1,Bobby Portis,Dominick Barlow,Will Richard,14.5,5.5,6.5,9.42,8.15,11.10,under,over,over,0,12.14,0.243,High,Med,High
2,Dominick Barlow,Deni Avdija,Donovan Clingan,5.5,23.5,8.5,8.15,29.23,11.46,over,over,over,0,9.23,0.185,Med,High,High
3,Tyus Jones,Kobe Sanders,Donovan Clingan,3.5,9.5,8.5,4.46,12.70,11.46,over,over,over,0,6.92,0.138,Low,High,High
4,Tyus Jones,Kobe Sanders,Brandin Podziemski,3.5,9.5,11.5,4.46,12.70,14.85,over,over,over,0,6.61,0.132,Low,High,High


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 68 players...
Error getting prediction for Paul George: float division by zero
Processing 61 players with valid predictions...
Generated 34757 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 40 combinations from 34757 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Bobby Portis,Deni Avdija,Will Richard,14.5,23.5,6.5,9.42,29.23,11.10,under,over,over,1,13.02,0.260,High,High,High
1,Bobby Portis,Donovan Clingan,Will Richard,14.5,8.5,6.5,9.42,11.46,11.10,under,over,over,0,11.24,0.225,High,High,High
2,Tyus Jones,Deni Avdija,Donovan Clingan,3.5,23.5,8.5,4.46,29.23,11.46,over,over,over,0,8.40,0.168,Low,High,High
3,Kobe Sanders,Tyus Jones,Brandin Podziemski,9.5,3.5,11.5,12.70,4.46,14.85,over,over,over,0,6.61,0.132,High,Low,High
4,Kobe Sanders,Goga Bitadze,Brandin Podziemski,9.5,4.5,11.5,12.70,5.94,14.85,over,over,over,0,6.38,0.128,High,Low,High
